# Voronoi Topology Descriptors

Beyond the Voronoi signature vector `(n3, n4, n5, n6)`, pyscal3 now
provides additional topological measures of the Voronoi cell:

* **Cell volume** — volume of the Voronoi polyhedron
* **Sphericity** (isoperimetric quotient) — $IQ = 36\pi V^2 / A^3$,
  measuring how spherical the cell is (IQ=1 for a sphere)
* **Face analysis** — per-face statistics (count, area, perimeter)

In [1]:
import numpy as np
import pyscal3 as pc
from ase.build import bulk

## 1. Cell volume

In [2]:
al = bulk("Al", cubic=True).repeat(3)
pc.find_neighbors(al, method="voronoi")

vol = pc.voronoi_cell_volume(al)
expected = al.cell.volume / len(al)
print("Per-atom cell volume: {:.3f} A^3".format(vol[0]))
print("Expected (V/N):      {:.3f} A^3".format(expected))

Per-atom cell volume: 16.608 A^3
Expected (V/N):      16.608 A^3


## 2. Sphericity

In [3]:
for name, struct in [("FCC Al", "Al"), ("BCC Fe", "Fe"), ("HCP Mg", None)]:
    if struct:
        atoms = bulk(struct, cubic=True).repeat(3)
    else:
        atoms = bulk("Mg", "hcp", a=3.21, c=5.21).repeat((3, 3, 3))
    pc.find_neighbors(atoms, method="voronoi")
    iq = pc.voronoi_sphericity(atoms)
    print("{}: IQ = {:.4f}".format(name, iq[0]))

FCC Al: IQ = 0.7405
BCC Fe: IQ = 0.7534
HCP Mg: IQ = 0.6520


## 3. Face analysis

The face analysis returns per-atom statistics about the individual faces:
number of faces, mean/std/max face area, mean perimeter.

In [4]:
fe = bulk("Fe", cubic=True).repeat(3)
pc.find_neighbors(fe, method="voronoi")
fa = pc.voronoi_face_analysis(fe)
print("BCC Fe per atom:")
print("  Number of faces:", fa["n_faces"][0])
print("  Mean face area:  {:.3f} A^2".format(fa["mean_face_area"][0]))
print("  Std face area:   {:.3f} A^2".format(fa["std_face_area"][0]))
print("  Max face area:   {:.3f} A^2".format(fa["max_face_area"][0]))
print("  Mean perimeter:  {:.3f} A".format(fa["mean_face_perimeter"][0]))

BCC Fe per atom:
  Number of faces: 14
  Mean face area:  1.970 A^2
  Std face area:   0.814 A^2
  Max face area:   2.675 A^2
  Mean perimeter:  5.218 A


## 4. Comparing structures

| Property | FCC (rhombic dodecahedron) | BCC (truncated octahedron) |
|----------|--------------------------|---------------------------|
| Faces | 12 | 14 |
| Signature | (0, 12, 0, 0) | (0, 6, 0, 8) |
| Sphericity | ~0.74 | ~0.91 |

In [5]:
print("{:<8s}  {:>8s}  {:>8s}  {:>8s}".format("Struct", "N_faces", "IQ", "Volume"))
for name, struct, kw in [("FCC Al", "Al", {}), ("BCC Fe", "Fe", {}),
                          ("HCP Mg", "Mg", {"crystalstructure": "hcp", "a": 3.21, "c": 5.21})]:
    if kw:
        atoms = bulk(struct, **kw).repeat((3, 3, 3))
    else:
        atoms = bulk(struct, cubic=True).repeat(3)
    pc.find_neighbors(atoms, method="voronoi")
    vol = pc.voronoi_cell_volume(atoms)
    iq = pc.voronoi_sphericity(atoms)
    fa = pc.voronoi_face_analysis(atoms)
    print("{:<8s}  {:>8d}  {:>8.4f}  {:>8.3f}".format(
        name, fa["n_faces"][0], iq[0], vol[0]))

Struct     N_faces        IQ    Volume
FCC Al          12    0.7405    16.608
BCC Fe          14    0.7534    11.820
HCP Mg          14    0.6520    40.440
